# Phase 8 — Hyperparameter Tuning

**What:** tune the two best model families from Phase 7 (Gradient Boosting and XGBoost) so
their hyperparameters fit *this* problem instead of scikit-learn defaults.

**Why:** defaults are "safe for everything, best for nothing". Tuning finds the sweet spot —
but it must not fool us. Our protocol stays honest:

1. **Search on the training set only**, with 3-fold cross-validation, maximizing **ROC-AUC**
   (threshold-free, so the search isn't biased by the default 0.5 cutoff).
2. With the best hyperparameters, pick the decision threshold on training **out-of-fold**
   predictions (same Phase 6 protocol).
3. Evaluate on the test set **exactly once**.

## 1. Setup

The tuning logic lives in `src/tune.py` so the notebook and the command-line script share
one implementation. We only import and call it here.

In [1]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
if (cwd / "src").exists():
    BASE = cwd
elif (cwd.parent / "src").exists():
    BASE = cwd.parent
else:
    BASE = cwd
sys.path.insert(0, str(BASE / "src"))
sys.path.insert(0, str(BASE))

from split_data import load_splits
from tune import search
from evaluate import evaluate_model, find_best_threshold

X_train, X_test, y_train, y_test = load_splits(BASE / "data" / "processed" / "splits")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (12156, 16) | X_test: (3040, 16)


## 2. Gradient Boosting — hyperparameter search

Randomized search over the most influential knobs:

| Parameter | Meaning | Range tried |
|-----------|---------|-------------|
| `n_estimators` | number of trees | 200–500 |
| `max_depth` | tree depth (interactions) | 3–6 |
| `learning_rate` | step size per tree | 0.03–0.1 |
| `subsample` | fraction of rows per tree | 0.7–1.0 |
| `min_samples_leaf` | minimum samples in a leaf (regularization) | 5–20 |

Each of the 30 candidate combos is scored by 3-fold CV ROC-AUC.

In [2]:
gb_search = search("GradientBoosting", X_train, y_train, n_iter=10)


=== Searching GradientBoosting (10 random combos x 3 folds) ===


Best CV ROC-AUC : 0.8891
Best params     : {'classifier__learning_rate': 0.03, 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 20, 'classifier__n_estimators': 369, 'classifier__subsample': 1.0}
Search took     : 797.3s


## 3. XGBoost — hyperparameter search

Same protocol for XGBoost, with its own parameters (`colsample_bytree`, `min_child_weight`
plus `scale_pos_weight=9` to counter the rain imbalance).

In [3]:
xgb_search = search("XGBoost", X_train, y_train, n_iter=12)


=== Searching XGBoost (12 random combos x 3 folds) ===


Best CV ROC-AUC : 0.8928
Best params     : {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.03, 'classifier__max_depth': 6, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 271, 'classifier__subsample': 0.7}
Search took     : 366.8s


## 4. Honest threshold tuning for the tuned models

The search maximized *ranking* quality (AUC). Now we find the F1-max decision threshold for
each tuned model using training out-of-fold predictions — the test set is still untouched.

In [4]:
gb_threshold = find_best_threshold(gb_search.best_estimator_, X_train, y_train)
xgb_threshold = find_best_threshold(xgb_search.best_estimator_, X_train, y_train)
print(f"GradientBoosting (tuned) threshold: {gb_threshold:.2f}")
print(f"XGBoost (tuned)          threshold: {xgb_threshold:.2f}")

GradientBoosting (tuned) threshold: 0.25
XGBoost (tuned)          threshold: 0.75


## 5. Final comparison — test set evaluated once

One-time test evaluation. The Phase 7 Gradient Boosting result is included as a recorded
reference row (not recomputed) to show what tuning bought us.

In [5]:
rows = [
    {"model": "GB (Phase 7, recorded)", "threshold": 0.25,
     "precision": 0.734, "recall": 0.752, "f1": 0.743},
]
for name, pipe, thr in [
    ("GradientBoosting (tuned)", gb_search.best_estimator_, gb_threshold),
    ("XGBoost (tuned)", xgb_search.best_estimator_, xgb_threshold),
]:
    m = evaluate_model(pipe, X_test, y_test, threshold=thr)
    rows.append({"model": name, "threshold": thr,
                 **{k: round(v, 3) for k, v in m.items()}})

final = pd.DataFrame(rows)[["model", "threshold", "accuracy", "precision", "recall", "f1"]]
final = final.sort_values("f1", ascending=False).reset_index(drop=True)
final

,model,threshold,accuracy,precision,recall,f1
0,"GB (Phase 7, recorded)",0.25,NaN,0.734,0.752,0.743
1,GradientBoosting (tuned),0.25,0.863,0.718,0.740,0.729
2,XGBoost (tuned),0.75,0.865,0.762,0.666,0.711


## 6. Pick the champion and save it

The champion is the best of the *tuned* models by F1 (rain-catching matters most). The
artifact saved to `models/karachi_rain_model.pkl` carries the pipeline **and** its threshold,
so the API and dashboard will use the exact same decision rule.

In [6]:
import joblib

champion = final.iloc[0] if "tuned" in final.iloc[0]["model"] else final.iloc[1]
name = champion["model"]
pipeline = gb_search.best_estimator_ if "GradientBoosting" in name else xgb_search.best_estimator_
threshold = gb_threshold if "GradientBoosting" in name else xgb_threshold

MODEL_PATH = BASE / "models" / "karachi_rain_model.pkl"
joblib.dump({"model_name": name, "threshold": float(threshold), "pipeline": pipeline}, MODEL_PATH)
print(f"Champion: {name} | F1 = {champion['f1']:.3f} @ threshold {threshold:.2f}")
print("Saved ->", MODEL_PATH)

Champion: GradientBoosting (tuned) | F1 = 0.729 @ threshold 0.25
Saved -> G:\Machine Learning Models\1st Model\models\karachi_rain_model.pkl


## Takeaway

The search did **not** beat the Phase 7 hand-picked defaults (best tuned test F1 0.729
vs. the Phase 7 GradientBoosting 0.743). Randomized search found better **ranking** on the
training CV (GB 0.889, XGB 0.893 AUC), but that gain did not survive the one-time test
evaluation. The Phase 7 champion was therefore **restored** as `models/karachi_rain_model.pkl`
(GradientBoosting, threshold 0.25, test F1 0.743) via `python src/train.py`, so the deployed
artifact stays the best model we have. This is the honest outcome: on this small, noisy
dataset the defaults were already close to the optimum, and tuning was not worth the risk
of overfitting the validation procedure.